[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C48_Cloud_Deployment_Course/03_kubernetes/03_kubernetes.ipynb)

# 03 · Kubernetes 编排（把控制循环、探针状态机与调度器从零写出来）

目标：把 **reconcile 控制循环 → 探针状态机 → 端点管理 → 装箱调度 → 碎片量化** 从零实现，
每个机制都**对拍**朴素参考、每个策略都**算一笔账**。

路线：Deployment 控制器 → 探针 FSM（三种探针的时序）→ 端点控制器与流量摘除 → first-fit/best-fit/装箱 → 碎片 vs 反亲和 → ✏️ 练习 → 📖 答案 → 🧪 集群成本胶囊。

> 心智模型：**K8s = 一堆水平触发的幂等控制循环 + 一个装箱调度器**。
> 本环境没有集群，但这两样东西的全部逻辑都是纯计算。

## 1 · 控制循环：水平触发为什么能容错

模块 00 写过最小 reconcile。这里加上**水平触发（level-triggered）**的关键性质验证：
控制器**只看当前状态、不看事件历史**，所以丢消息、重复、乱序都不会让它错位。

对照组是**边沿触发（edge-triggered）**：「收到创建事件就创建」——丢一条就永久错位。

In [ ]:
import math, random, heapq
from dataclasses import dataclass, field

def level_triggered(current_count, desired, events_lost=0):
    '''水平触发：无视事件，只比较状态。'''
    return desired - current_count            # 直接算差量

def edge_triggered(current_count, event_queue):
    '''边沿触发：按事件逐条响应。丢事件 = 永久错位。'''
    delta = 0
    for ev in event_queue:
        delta += 1 if ev == 'scale_up' else -1
    return delta

DESIRED = 5
# 场景：期望从 0 扩到 5，但有 2 条事件在网络里丢了
events = ['scale_up'] * 5
lossy  = events[:3]                            # 丢了 2 条

lt = level_triggered(0, DESIRED)
et = edge_triggered(0, lossy)
print(f'水平触发算出要创建 {lt} 个 | 边沿触发（丢2条）算出 {et} 个')
assert lt == 5, '水平触发不受丢消息影响'
assert et == 3, '边沿触发丢了 2 条就永久少 2 个'

# 重复投递同样安全
assert level_triggered(0, DESIRED) == level_triggered(0, DESIRED), '幂等：重放安全'
assert edge_triggered(0, events + events) == 10, '边沿触发重复投递会翻倍！'
print('✅ 水平触发对「丢失/重复/乱序」全免疫 —— 这是 K8s 所有控制器的基石')

### 多个控制器协作：Deployment → ReplicaSet → Pod

真实 K8s 是**多层循环**：Deployment 管 ReplicaSet，ReplicaSet 管 Pod。
每层独立、异步、最终一致。下面把两层串起来，验证整体仍然收敛。

In [ ]:
@dataclass
class Pod:
    name: str
    rs: str
    phase: str = 'Pending'      # Pending -> Running
    ready: bool = False

class Cluster:
    def __init__(self):
        self.pods, self.rs_desired, self._n = [], {}, 0
    # ── ReplicaSet 控制器 ──
    def rs_reconcile(self):
        acts = []
        for rs, want in self.rs_desired.items():
            have = [p for p in self.pods if p.rs == rs]
            for _ in range(want - len(have)):
                self._n += 1
                self.pods.append(Pod(f'pod-{self._n}', rs)); acts.append(('create', rs))
            for p in have[want:]:
                self.pods.remove(p); acts.append(('delete', p.name))
        return acts
    # ── kubelet：把 Pending 推进到 Running/Ready ──
    def kubelet_tick(self):
        for p in self.pods:
            if p.phase == 'Pending':   p.phase = 'Running'
            elif not p.ready:          p.ready = True

c = Cluster()
c.rs_desired = {'rs-v1': 3}
trace = []
for _ in range(4):
    c.rs_reconcile(); c.kubelet_tick()
    trace.append((len(c.pods), sum(p.ready for p in c.pods)))
print('(总数, ready 数) 演化:', trace)
assert trace[-1] == (3, 3), '两层循环最终都收敛'
# 关键：总数先到位，ready 后到位 —— 最终一致，不是原子
assert trace[0][0] == 3 and trace[0][1] == 0, '第一轮 Pod 已创建但还没 ready'
print('✅ 「Pod 存在」≠「Pod 就绪」≠「流量已切过去」—— 每一跳都是独立循环')

## 2 · 探针状态机：三种探针的时序契约

**startupProbe 成功之前，liveness 与 readiness 都不执行。** 这条契约是 LLM 服务的救命稻草：
没有它，慢启动的模型会被 liveness 反复杀死，陷入永远加载不完的重启死循环。

In [ ]:
@dataclass
class ProbeCfg:
    period: int
    failure_threshold: int
    initial_delay: int = 0

@dataclass
class PodRuntime:
    load_seconds: int                 # 模型加载需要多久
    t: int = 0
    started: bool = False
    alive: bool = True
    ready: bool = False
    restarts: int = 0
    startup_fails: int = 0
    live_fails: int = 0

def tick(pod, startup: ProbeCfg, live: ProbeCfg, ready: ProbeCfg, use_startup=True):
    '''推进 1 秒，执行探针语义。返回本秒发生的事件。'''
    pod.t += 1
    loaded = pod.t >= pod.load_seconds
    ev = None
    if use_startup and not pod.started:
        if pod.t % startup.period == 0:
            if loaded:
                pod.started = True; ev = 'startup-ok'
            else:
                pod.startup_fails += 1
                if pod.startup_fails >= startup.failure_threshold:
                    ev = 'RESTART(startup)'; pod.restarts += 1
                    pod.t = 0; pod.startup_fails = 0
        return ev
    # startup 已通过（或未启用）：liveness + readiness 生效
    if pod.t % live.period == 0:
        if loaded:  pod.live_fails = 0
        else:
            pod.live_fails += 1
            if pod.live_fails >= live.failure_threshold:
                ev = 'RESTART(liveness)'; pod.restarts += 1
                pod.t = 0; pod.live_fails = 0; pod.started = False
                return ev
    if pod.t % ready.period == 0:
        pod.ready = loaded
    return ev

LOAD = 180      # 模型加载 3 分钟
S = ProbeCfg(period=10, failure_threshold=60)     # 600s 启动窗口
L = ProbeCfg(period=20, failure_threshold=3)      # 60s 发现假死
R = ProbeCfg(period=5,  failure_threshold=2)

# 情况 A：正确配置（有 startupProbe）
a = PodRuntime(load_seconds=LOAD)
for _ in range(400): tick(a, S, L, R, use_startup=True)
print(f'有 startupProbe : 重启 {a.restarts} 次, ready={a.ready}')
assert a.restarts == 0 and a.ready, '正确配置下应零重启并最终就绪'

# 情况 B：没有 startupProbe，liveness 直接生效
b = PodRuntime(load_seconds=LOAD)
for _ in range(400): tick(b, S, L, R, use_startup=False)
print(f'无 startupProbe : 重启 {b.restarts} 次, ready={b.ready}')
assert b.restarts >= 3, '没有 startupProbe，慢启动模型会被反复杀死'
assert not b.ready, '陷入重启死循环，永远加载不完'
print('\n✅ 这就是「模型越大越起不来」的真实原因 —— 不是模型的问题，是探针配置的问题')

### readiness 决定流量：端点控制器

**Service 的端点列表 = 所有 ready 的 Pod。** 这就是 readiness 的全部意义。
下面把端点控制器写出来，复现模块 02 提到的头号事故：readiness 恒返回 200。

In [ ]:
def endpoints(pods):
    '''端点控制器：只收录 ready 的 Pod。'''
    return [p.name for p in pods if p.ready and p.phase == 'Running']

def route(pods, n_requests, rng):
    '''把请求分给端点；打到未就绪 Pod 的请求全部失败。'''
    eps = endpoints(pods)
    if not eps:
        return 0, n_requests            # 无端点：全部失败（503）
    ok = fail = 0
    for _ in range(n_requests):
        target = rng.choice([p for p in pods if p.name in eps])
        if target.ready: ok += 1
        else:            fail += 1
    return ok, fail

rng = random.Random(0)
# 正确：readiness 反映真实状态
good_pods = [Pod('p1','rs',phase='Running',ready=True),
             Pod('p2','rs',phase='Running',ready=False)]   # p2 还在加载
ok, fail = route(good_pods, 1000, rng)
assert fail == 0 and ok == 1000, '未就绪的 p2 不在端点里，流量全部安全'
print(f'正确 readiness: 端点 {endpoints(good_pods)} -> 成功 {ok}, 失败 {fail}')

# 错误：readiness 恒 200（图省事），未加载完的 Pod 也进端点
class FakeReadyPod(Pod):
    pass
bad_pods = [Pod('p1','rs',phase='Running',ready=True),
            Pod('p2','rs',phase='Running',ready=True)]     # ← 谎报就绪
actually_loaded = {'p1': True, 'p2': False}
eps = endpoints(bad_pods)
ok = sum(1 for _ in range(1000) if actually_loaded[rng.choice(eps)])
print(f'错误 readiness: 端点 {eps} -> 成功 {ok}/1000 (约一半请求打到未加载的 Pod)')
assert 400 < ok < 600, '约一半流量会失败'
print('\n✅ 复现了 LLM 部署的头号事故：Pod 全是 Running，监控一切正常，服务半死不活')

## 3 · 调度器：过滤 + 打分 + 装箱

调度 = **过滤**（硬约束）+ **打分**（软偏好）。
下面实现两种打分策略，量化它们对**碎片**的影响。

In [ ]:
@dataclass
class Node:
    name: str
    gpu: int; cpu: int; mem: int
    used_gpu: int = 0; used_cpu: int = 0; used_mem: int = 0
    zone: str = 'a'
    def free(self):  return (self.gpu-self.used_gpu, self.cpu-self.used_cpu, self.mem-self.used_mem)
    def fits(self, req):
        f = self.free(); return all(f[i] >= req[i] for i in range(3))
    def place(self, req):
        self.used_gpu += req[0]; self.used_cpu += req[1]; self.used_mem += req[2]

def schedule(nodes, req, policy='least'):
    '''返回被选中的节点，或 None。'''
    feasible = [n for n in nodes if n.fits(req)]          # 阶段一：过滤
    if not feasible: return None
    def score(n):                                          # 阶段二：打分
        used_ratio = n.used_gpu / n.gpu if n.gpu else 0
        return -used_ratio if policy == 'least' else used_ratio
    return max(feasible, key=score)

def make_cluster(n_nodes=20):
    return [Node(f'n{i}', gpu=8, cpu=96, mem=768, zone='abc'[i % 3]) for i in range(n_nodes)]

# 混合负载：大量 1 卡小任务 + 少量 4 卡大任务
rng = random.Random(7)
workload = [(1, 8, 64)] * 60 + [(2, 16, 128)] * 10

results = {}
for policy in ['least', 'most']:
    nodes = make_cluster()
    placed = 0
    for req in workload:
        n = schedule(nodes, req, policy)
        if n: n.place(req); placed += 1
    total_free_gpu = sum(n.free()[0] for n in nodes)
    max_contiguous = max(n.free()[0] for n in nodes)       # 最大的单节点空闲
    empty_nodes = sum(1 for n in nodes if n.used_gpu == 0)
    results[policy] = (placed, total_free_gpu, max_contiguous, empty_nodes)
    print(f'{policy:>6s}: 放置 {placed}/{len(workload)}, 总空闲 {total_free_gpu} GPU, '
          f'最大单节点空闲 {max_contiguous}, 完整空节点 {empty_nodes}')

assert results['most'][3] > results['least'][3], '装箱策略应留下更多完整空节点'
assert results['most'][2] >= results['least'][2], '装箱策略的最大连续空闲应不小于分散策略'
print('\n✅ 两种策略总空闲卡数可能相同，但「能不能放下一个 8 卡任务」天差地别 —— 这就是碎片')

### 碎片的代价：总空闲 30 张卡，却放不下一个 4 卡任务

In [ ]:
def can_place(nodes, req):
    return any(n.fits(req) for n in nodes)

for policy in ['least', 'most']:
    nodes = make_cluster()
    for req in workload:
        n = schedule(nodes, req, policy)
        if n: n.place(req)
    free = sum(n.free()[0] for n in nodes)
    print(f'{policy:>6s} 策略, 总空闲 {free} GPU:')
    for size in [1, 2, 4, 8]:
        ok = can_place(nodes, (size, size*8, size*64))
        print(f'    还能放下 {size} 卡任务? {"✅" if ok else "❌ 放不下（碎片）"}')

nodes_least = make_cluster()
for req in workload:
    n = schedule(nodes_least, req, 'least')
    if n: n.place(req)
nodes_most = make_cluster()
for req in workload:
    n = schedule(nodes_most, req, 'most')
    if n: n.place(req)
assert can_place(nodes_most, (8, 64, 512)), '装箱策略应仍能放下整节点任务'
print('\n✅ 「集群还有 30 张卡空闲」和「能不能跑一个 8 卡任务」是两个完全不同的问题。')
print('   GPU 集群默认的 LeastAllocated 往往是错的 —— 它优化的是均衡，不是可用性。')

### 大 Pod 优先：避免资源搁浅（resource stranding）

在线贪心装箱有个经典结论：**按需求降序放置（first-fit-decreasing）接近最优**。
反过来，小任务先来会把每台机器都占一点，大任务永远排不进去。

In [ ]:
def run_order(order_workload, policy='least'):     # 'least' 是 K8s 的默认打分策略
    nodes = make_cluster(10)
    placed = []
    for req in order_workload:
        n = schedule(nodes, req, policy)
        if n: n.place(req); placed.append(req)
    return len(placed), len(order_workload)

mixed = [(1,8,64)] * 30 + [(8,64,512)] * 5        # 30 个 1 卡 + 5 个整节点任务
small_first = mixed
big_first   = sorted(mixed, key=lambda r: -r[0])   # FFD：按需求降序

p_small, tot = run_order(small_first)
p_big,   _   = run_order(big_first)
print(f'小任务先来 (随到随调度): 放下 {p_small}/{tot}')
print(f'大任务先来 (FFD/优先级): 放下 {p_big}/{tot}')
assert p_big > p_small, 'FFD 应放下更多任务'
print(f'\n✅ 同样的负载、同样的集群，只是顺序不同 -> 多放下 {p_big-p_small} 个任务。')
print('   这就是 PriorityClass 存在的理由：让大任务先决定位置。')

## ✏️ 练习 1：QoS class 判定

实现 `qos_class(containers)`：`containers` 是 `[{'req': {...}, 'lim': {...}}, ...]`。
规则（K8s 原文语义）：
- 所有容器的**所有**资源都设了 requests 且 requests == limits → `'Guaranteed'`
- 所有容器**都没设**任何 requests 和 limits → `'BestEffort'`
- 其余 → `'Burstable'`

In [ ]:
def qos_class(containers):
    # TODO: 按上述三条规则返回 'Guaranteed' / 'Burstable' / 'BestEffort'
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
g = [{'req': {'cpu': 4, 'memory': 8}, 'lim': {'cpu': 4, 'memory': 8}}]
b = [{'req': {'cpu': 2, 'memory': 4}, 'lim': {'cpu': 4, 'memory': 8}}]
e = [{'req': {}, 'lim': {}}]
mixed = [{'req': {'cpu': 4}, 'lim': {'cpu': 4}}, {'req': {}, 'lim': {}}]
assert qos_class(g) == 'Guaranteed'
assert qos_class(b) == 'Burstable'
assert qos_class(e) == 'BestEffort'
assert qos_class(mixed) == 'Burstable', '只要有一个容器不满足，整个 Pod 就降档'
# 只设了 cpu 没设 memory -> 不是 Guaranteed
partial = [{'req': {'cpu': 4}, 'lim': {'cpu': 4, 'memory': 8}}]
assert qos_class(partial) == 'Burstable'
print('✅ 练习 1 通过：生产 LLM 服务必须是 Guaranteed（requests == limits 且都设全）')

## ✏️ 练习 2：拓扑均匀分布

实现 `spread_place(nodes, n_replicas, req, max_skew=1)`：
把 `n_replicas` 个副本放到 `nodes` 上，要求**各 zone 的副本数最大差值 ≤ max_skew**
（K8s 的 `topologySpreadConstraints`）。每次挑「当前副本数最少的 zone 里、能放下的、已用最多的节点」（区内装箱）。
返回被选中的节点名列表；放不下就跳过该副本。

In [ ]:
def spread_place(nodes, n_replicas, req, max_skew=1):
    # TODO: 维护 per-zone 计数；每轮挑计数最小的 zone；
    #       在该 zone 内选 fits 且 used_gpu 最大的节点（区内装箱）；放不下则试下一个 zone
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
ns = make_cluster(9)                 # 9 节点，zone a/b/c 各 3 个
chosen = spread_place(ns, 6, (2, 16, 128))
assert len(chosen) == 6, f'应放下 6 个副本，实际 {len(chosen)}'
byzone = {}
for name in chosen:
    z = next(n.zone for n in ns if n.name == name)
    byzone[z] = byzone.get(z, 0) + 1
print('各 zone 副本数:', byzone)
assert max(byzone.values()) - min(byzone.values()) <= 1, f'zone 间偏斜应 ≤1，实际 {byzone}'
assert len(byzone) == 3, '应铺满 3 个 zone'
# 区内装箱：使用中的节点数应少于副本数（说明有节点放了 >1 个副本）
used_nodes = len(set(chosen))
assert used_nodes <= 6
print(f'✅ 练习 2 通过：跨 AZ 均匀（防区域故障）+ 区内装箱（省钱），用了 {used_nodes} 个节点')

## ✏️ 练习 3：PodDisruptionBudget

实现 `can_evict(total, ready, min_available)`：节点维护时要驱逐一个 Pod，
只有在**驱逐后仍满足 `ready - 1 >= min_available`** 时才允许。返回 bool。
再实现 `drain_node(pods_on_node, total, ready, min_available)`：
按顺序尝试驱逐，返回**实际能驱逐的个数**。

In [ ]:
def can_evict(total, ready, min_available):
    # TODO
    raise NotImplementedError

def drain_node(pods_on_node, total, ready, min_available):
    # TODO: 逐个尝试；每成功驱逐一个，ready 减 1
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert can_evict(total=10, ready=10, min_available=8) is True
assert can_evict(total=10, ready=8,  min_available=8) is False, '已到下限，不能再驱逐'
n = drain_node(pods_on_node=4, total=10, ready=10, min_available=8)
assert n == 2, f'10 个 ready、下限 8 -> 只能驱逐 2 个，得到 {n}'
n2 = drain_node(pods_on_node=4, total=10, ready=10, min_available=10)
assert n2 == 0, 'min_available == 副本数 -> 一个都不能驱逐（节点永远排空不了！）'
print('✅ 练习 3 通过：PDB 保护可用性，但设得太严会让节点维护永远卡住')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def qos_class(containers):
    all_empty = all(not c['req'] and not c['lim'] for c in containers)
    if all_empty:
        return 'BestEffort'
    guaranteed = True
    for c in containers:
        req, lim = c['req'], c['lim']
        if not req or not lim or set(req) != set(lim) or any(req[k] != lim[k] for k in req):
            guaranteed = False; break
    return 'Guaranteed' if guaranteed else 'Burstable'

In [ ]:
# 练习 2 参考答案
def spread_place(nodes, n_replicas, req, max_skew=1):
    zones = sorted({n.zone for n in nodes})
    count = {z: 0 for z in zones}
    chosen = []
    for _ in range(n_replicas):
        placed = False
        for z in sorted(zones, key=lambda z: count[z]):
            cand = [n for n in nodes if n.zone == z and n.fits(req)]
            if not cand:
                continue
            node = max(cand, key=lambda n: n.used_gpu)      # 区内装箱
            node.place(req); count[z] += 1; chosen.append(node.name); placed = True
            break
        if not placed:
            break
    return chosen

In [ ]:
# 练习 3 参考答案
def can_evict(total, ready, min_available):
    return ready - 1 >= min_available

def drain_node(pods_on_node, total, ready, min_available):
    evicted = 0
    for _ in range(pods_on_node):
        if not can_evict(total, ready, min_available):
            break
        ready -= 1; evicted += 1
    return evicted

---
## 🧪 真实数据胶囊：反亲和的溢价到底值不值

把「装箱 vs 反亲和 vs 跨 AZ 折中」三种策略的成本与可用性放在一起算。
（公开量级：8 卡节点 $32/h；单机年故障率 ~2%；恢复 10 分钟。）

In [ ]:
REPLICAS, GPU_PER_REPLICA, GPU_PER_NODE, NODE_HOURLY = 13, 2, 8, 32.0
ANNUAL_NODE_FAILURE, MTTR_MIN = 0.02, 10.0

def strategy_cost(nodes_needed, replicas_per_node):
    monthly = nodes_needed * NODE_HOURLY * 24 * 30
    packing = REPLICAS * GPU_PER_REPLICA / (nodes_needed * GPU_PER_NODE)
    capacity_loss = replicas_per_node / REPLICAS          # 一台机器挂了损失多少容量
    # 年期望不可用分钟（按容量损失折算）
    expected_min = nodes_needed * ANNUAL_NODE_FAILURE * MTTR_MIN * capacity_loss
    return monthly, packing, capacity_loss, expected_min

strategies = [
    ('理想装箱 (4 副本/节点)',  math.ceil(REPLICAS / 4), 4),
    ('跨3AZ + 区内装箱',        6,                       3),
    ('硬反亲和 (1 副本/节点)',  REPLICAS,                1),
]
print(f"{'策略':<24s} {'节点':>5s} {'装箱率':>7s} {'单机故障损失':>12s} {'月成本$':>10s} {'年期望损失分钟':>14s}")
rows = []
for name, nodes, rpn in strategies:
    m, pk, cl, em = strategy_cost(nodes, rpn)
    rows.append((name, nodes, m, em))
    print(f'{name:<24s} {nodes:>5d} {pk:>7.0%} {cl:>11.1%} {m:>10,.0f} {em:>14.2f}')

cheap, mid, expensive = rows[0][2], rows[1][2], rows[2][2]
loss_cheap, loss_expensive = rows[0][3], rows[2][3]
premium = expensive - cheap
saved_min = loss_cheap - loss_expensive
print(f'\n硬反亲和比装箱贵 ${premium:,.0f}/月，换来年期望少损失 {saved_min:.2f} 分钟')
print(f'折合每减少 1 分钟年不可用，要花 ${premium*12/max(saved_min,1e-9):,.0f}')
assert expensive > 3 * cheap, '硬反亲和的溢价超过 3 倍'
assert mid < expensive, '跨 AZ 折中应显著便宜于硬反亲和'
print('\n✅ 结论：硬反亲和的溢价通常不划算；「跨 AZ 均分 + 区内装箱」几乎总是正确答案。')

**🧪 胶囊练习**：实现 `fragmentation(nodes)`：返回 `(总空闲GPU, 最大可容纳的单任务GPU数, 碎片率)`。
碎片率定义为 `1 - 最大可容纳 / 总空闲`（总空闲为 0 时返回 0.0）。

In [ ]:
def fragmentation(nodes):
    # TODO: total = Σ 各节点空闲 GPU；largest = max(各节点空闲 GPU)
    #       frag = 0.0 if total == 0 else 1 - largest / total
    raise NotImplementedError

In [ ]:
# 自测
ns = make_cluster(4)
for n in ns: n.place((7, 8, 64))            # 每节点占 7 卡，各剩 1 卡
total, largest, frag = fragmentation(ns)
assert total == 4 and largest == 1, (total, largest)
assert abs(frag - 0.75) < 1e-9, f'4 张空闲卡分散在 4 台机器上，碎片率应为 0.75，得到 {frag}'

ns2 = make_cluster(4)
ns2[0].place((8, 8, 64)); ns2[1].place((8, 8, 64)); ns2[2].place((8, 8, 64))
total2, largest2, frag2 = fragmentation(ns2)
assert total2 == 8 and largest2 == 8 and frag2 == 0.0, '集中在一台机器 -> 零碎片'
print(f'分散: 空闲 {total}, 最大 {largest}, 碎片率 {frag:.0%}')
print(f'装箱: 空闲 {total2}, 最大 {largest2}, 碎片率 {frag2:.0%}')
print('✅ 胶囊练习通过：同样 4~8 张空闲卡，碎片率决定了「能不能真的用上」')

In [ ]:
# 📖 胶囊参考答案
def fragmentation(nodes):
    frees = [n.free()[0] for n in nodes]
    total, largest = sum(frees), max(frees) if frees else 0
    return total, largest, (0.0 if total == 0 else 1 - largest / total)

---
## 🔧 旁注：真实系统里这些对应什么

- **水平触发控制循环** → controller-runtime 的 `Reconcile(ctx, req)`；你写 Operator 时实现的就是这个函数。
- **探针状态机** → kubelet 的 prober manager；`kubectl describe pod` 里的 `Liveness/Readiness/Startup` 行与 `Events` 里的 `Unhealthy`。
- **端点控制器** → EndpointSlice controller；`kubectl get endpointslices` 能直接看到「谁在接流量」。排查「Pod Running 但没流量」第一步就看这个。
- **过滤+打分调度** → kube-scheduler 的 Filter/Score 插件框架；`NodeResourcesFit` 的 `scoringStrategy` 就是 `LeastAllocated`/`MostAllocated` 的开关。
- **FFD / 大 Pod 优先** → `PriorityClass` + 抢占；GPU 集群常配 Volcano/Koordinator 做 gang scheduling。
- **PDB** → `PodDisruptionBudget`；`kubectl drain` 卡住十有八九是 PDB 设得太严。

你在这里写的调度器只有几十行，但它和 kube-scheduler 的**结构**是一致的：过滤、打分、选最高分。差别在插件数量，不在思想。

### 小结
- K8s 的核心只有一个：**水平触发的幂等控制循环**。它对丢消息/重复/乱序免疫，代价是**一切最终一致**。
- 对象是层层包裹的：Ingress → Service（端点=ready 的 Pod）→ Deployment → ReplicaSet → Pod → Node。
- **三种探针各管一段**：startupProbe 是 LLM 慢启动的救命稻草；liveness 只测自己；readiness 决定流量。
- **资源模型对 GPU 显存是盲的**：K8s 只数卡，显存预算必须在应用层算死。生产 Pod 应为 Guaranteed QoS。
- **调度是在线向量装箱**：GPU 集群默认的 LeastAllocated 制造碎片；大 Pod 优先（FFD）能显著提高放置率。
- 硬反亲和的可用性溢价通常不划算，**跨 AZ 均分 + 区内装箱**是标准答案。

下一站：**模块 04 · 发布策略与自动扩缩** —— 集群会自愈了，但「换个版本」和「流量涨十倍」还是两件会出事的事。